In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

spark = SparkSession.builder \
    .appName("DeltaSession") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate() 

spark.sparkContext.setLogLevel("ERROR")    

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/spark/.ivy2.5.2/cache
The jars for the packages stored in: /home/spark/.ivy2.5.2/jars
io.delta#delta-spark_2.13 added as a dependency
org.apache.spark#spark-connect_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-86990fa2-7bd9-40a3-b897-252cf0d98f67;1.0
	confs: [default]
	found io.delta#delta-spark_2.13;4.0.1 in central
	found io.delta#delta-storage;4.0.1 in central
	found org.antlr#antlr4-runtime;4.13.1 in central
	found org.apache.spark#spark-connect_2.13;4.0.1 in central
	found org.scala-lang.modules#scala-parallel-collections_2.13;1.2.0 in central
	found jakarta.servlet#jakarta.servlet-api;5.0.0 in central
	found javax.servlet#javax.servlet-api;4.0.1 in central
	found org.spark-project.spark#unused;1.0.0 in central
:: resolution report :: resolve 800ms :: artifacts dl 254ms
	:: modules in use:


In [ ]:
products_schema = StructType([
    StructField("product_id", IntegerType(), False),
    StructField("product_name", StringType(), False),
    StructField("category", StringType(), False),
    StructField("price", IntegerType(), False)
])
sales_schema = StructType([
        StructField("sale_id", IntegerType(), False),
        StructField("product_id", IntegerType(), False),
        StructField("quantity", IntegerType(), False),
        StructField("sale_date", StringType(), False)
    ])

def initialize_sample_data():
    products_data = [
        (101, "Laptop", "Electronics", 1200),
        (102, "Smartphone", "Electronics", 800),
        (103, "Desk Chair", "Furniture", 150),
        (104, "Monitor", "Electronics", 300),
        (105, "Backpack", "Travel", 50)
    ]
    products_df = spark.createDataFrame(products_data, schema=products_schema)

    sales_data = [
        (1, 101, 2, "2024-01-01"),
        (2, 102, 1, "2024-01-02"),
        (3, 101, 1, "2024-01-03"),
        (4, 103, 4, "2024-01-04"),
        (5, 105, 2, "2024-01-05"),
        (6, 999, 1, "2024-01-06") 
    ]
    sales_df = spark.createDataFrame(sales_data, schema=sales_schema)
    return products_df, sales_df

products_df, sales_df = initialize_sample_data()

products_df.write.format("delta").mode("overwrite").save("/opt/spark/delta/products")
sales_df.write.format("delta").mode("overwrite").save("/opt/spark/delta/sales")

In [ ]:
# question 1

products_df, sales_df = initialize_sample_data()

products_df.filter(F.col("price") >= 300) \
    .select("product_name", "category") \
    .orderBy(F.col("price").desc()).show()


+------------+-----------+
|product_name|   category|
+------------+-----------+
|      Laptop|Electronics|
|  Smartphone|Electronics|
|     Monitor|Electronics|
+------------+-----------+



In [ ]:
# question 2

joined = sales_df.join(products_df, "product_id", "inner")
joined.groupBy("category") \
    .agg(F.sum("quantity").alias("total_items")) \
        .orderBy("total_items").show()


+-----------+-----------+
|   category|total_items|
+-----------+-----------+
|     Travel|          2|
|Electronics|          4|
|  Furniture|          4|
+-----------+-----------+



In [ ]:
# question 3

products_df.withColumn("discounted_price", 
        F.when(F.col("category") == "Electronics", F.col("price") * 0.9)
              		.otherwise(F.col("price"))) \
              .filter(F.col("category") != "Travel") \
              .select("product_name", "discounted_price").show()


+------------+----------------+
|product_name|discounted_price|
+------------+----------------+
|      Laptop|          1080.0|
|  Smartphone|           720.0|
|  Desk Chair|           150.0|
|     Monitor|           270.0|
+------------+----------------+



In [ ]:
# question 4

products_df, sales_df = initialize_sample_data()
updates_schema = StructType([
    StructField("product_id", IntegerType(), False),
    StructField("new_price", IntegerType(), False)
])
updates_df = spark.createDataFrame([
    (102, 250),
    (106, 200)
], schema=updates_schema)

deltaTable = DeltaTable.forPath(spark, "/opt/spark/delta/products")

updates_df.show()

deltaTable.alias("target").merge(
    updates_df.alias("source"),
    "target.product_id = source.product_id"
).whenMatchedUpdate(set = {"price": "source.new_price"}) \
 .whenNotMatchedInsert(values = {
    "product_id": "source.product_id",
    "product_name": "'New Item'",
    "category": "'Misc'",
    "price": "source.new_price"
}).execute()

deltaTable.toDF().orderBy("product_id").show()

+----------+---------+
|product_id|new_price|
+----------+---------+
|       102|      250|
|       106|      200|
+----------+---------+

+----------+------------+-----------+-----+
|product_id|product_name|   category|price|
+----------+------------+-----------+-----+
|       101|      Laptop|Electronics| 1200|
|       102|  Smartphone|Electronics|  250|
|       103|  Desk Chair|  Furniture|  150|
|       104|     Monitor|Electronics|  300|
|       105|    Backpack|     Travel|   50|
|       106|    New Item|       Misc|  200|
+----------+------------+-----------+-----+



In [ ]:
deltaTable.toDF().write.format("delta").mode("overwrite").save("/opt/spark/delta/products")

In [ ]:
# question 5

products_df, sales_df = initialize_sample_data()

outer_df = products_df.join(sales_df, "product_id", "full")
outer_df.filter(F.col("product_name").isNull() | F.col("sale_id").isNull()) \
             .select("product_id", "product_name", "sale_id") \
             .orderBy("product_id").show()


+----------+------------+-------+
|product_id|product_name|sale_id|
+----------+------------+-------+
|       104|     Monitor|   NULL|
|       999|        NULL|      6|
+----------+------------+-------+



In [ ]:
# question 6

products_df.join(
    sales_df, 
    on="product_id", 
    how="left_anti"
).select("product_id", "product_name") \
    .orderBy("product_id").show()

+----------+------------+
|product_id|product_name|
+----------+------------+
|       104|     Monitor|
+----------+------------+

